# GeoHab 2026 — Rich Meta-Model + Simple Features + Seed Averaging

## Approach

Two-stage stacking pipeline:

**Stage 1 — Rich meta-model**: A LightGBM trained on the full 50+ feature set from the 6th place notebook (terrain derivatives, BPI, roughness, multi-scale neighbourhood stats, texture, backscatter texture, cross-features, spatial). Its OOF predicted probabilities (5 columns, one per class) are computed cleanly via cross-validation — no leakage.

**Stage 2 — Main seed-averaging model**: Trained on the original simple features (3×3 grid at ±12px + 8 basic stats = 26 features) **plus** the 5 meta-prob columns from Stage 1. This keeps the main model's feature space lean while injecting the rich terrain signal via the meta-features.

The main model learns when to trust the rich meta-signal and when to rely on its own direct observations.

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from scipy.ndimage import (
    uniform_filter, generic_filter, sobel, laplace, convolve
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

print('All imports OK ✓')

All imports OK ✓


## 1. Load Data

In [2]:
back     = rasterio.open('/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/backscatter.tif')
bath     = rasterio.open('/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/bathymetry.tif')
train_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/test.csv')

back_data = back.read(1)
bath_data = bath.read(1)

print(f'Train: {train_df.shape}  |  Test: {test_df.shape}')
print(f'Pixel size: {abs(bath.transform.a):.2f} m')
print('\nClass distribution:')
print(train_df['class'].value_counts())

Train: (6256, 3)  |  Test: (98, 3)
Pixel size: 0.25 m

Class distribution:
class
NVB     3036
FMAT    1544
SGZ      824
ALG      682
SGAM     170
Name: count, dtype: int64


## 2. Simple Features
Original features from the best-scoring notebook: 3×3 grid sampling at offsets ±12 pixels (18 grid features) + 8 basic neighbourhood stats.

In [3]:
def get_neighborhood_features(x, y):
    row, col = bath.index(x, y)
    features = []
    for dr in [-12, 0, 12]:
        for dc in [-12, 0, 12]:
            r, c = row + dr, col + dc
            try:
                depth   = bath_data[r, c]
                scatter = back_data[r, c]
            except:
                depth   = 0
                scatter = 0
            features.append(depth)
            features.append(scatter)
    return features

def get_advanced_features(x, y):
    row, col = bath.index(x, y)
    depths, scatters = [], []
    for dr in [-10, 0, 10]:
        for dc in [-10, 0, 10]:
            r, c = row + dr, col + dc
            if 0 <= r < bath_data.shape[0] and 0 <= c < bath_data.shape[1]:
                depths.append(bath_data[r, c])
                scatters.append(back_data[r, c])
            else:
                depths.append(0)
                scatters.append(0)
    return [
        np.mean(depths),   np.std(depths),
        np.min(depths),    np.max(depths),
        np.mean(scatters), np.std(scatters),
        np.min(scatters),  np.max(scatters),
    ]

# Extract for train
features = train_df.apply(lambda r: get_neighborhood_features(r['x'], r['y']), axis=1)
train_feature_cols = [f'f{i}' for i in range(len(features.iloc[0]))]
train_df[train_feature_cols] = pd.DataFrame(features.tolist(), index=train_df.index)

adv = train_df.apply(lambda r: get_advanced_features(r['x'], r['y']), axis=1)
adv_feature_cols = ['depth_mean','depth_std','depth_min','depth_max',
                    'scatter_mean','scatter_std','scatter_min','scatter_max']
train_df[adv_feature_cols] = pd.DataFrame(adv.tolist(), index=train_df.index)

# Extract for test
features = test_df.apply(lambda r: get_neighborhood_features(r['x'], r['y']), axis=1)
test_feature_cols = [f'f{i}' for i in range(len(features.iloc[0]))]
test_df[test_feature_cols] = pd.DataFrame(features.tolist(), index=test_df.index)

adv = test_df.apply(lambda r: get_advanced_features(r['x'], r['y']), axis=1)
test_df[adv_feature_cols] = pd.DataFrame(adv.tolist(), index=test_df.index)

SIMPLE_FEATURES = train_feature_cols + adv_feature_cols
print(f'Simple features: {len(SIMPLE_FEATURES)}')

Simple features: 26


## 3. Rich Features (for Meta-Model)
Full terrain pipeline from the 6th place notebook — computed on full raster arrays then sampled at point locations.

In [4]:
# ── Load and NaN-fill rasters ─────────────────────────────────────────────────
print('Loading raster arrays...')
bathy_arr = bath.read(1).astype(np.float32)
backs_arr = back.read(1).astype(np.float32)
bathy_arr[bathy_arr == bath.nodata] = np.nan
backs_arr[backs_arr == back.nodata] = np.nan

def nanmean_fill(arr):
    filled   = arr.copy()
    nan_mask = np.isnan(filled)
    if nan_mask.any():
        local_mean = uniform_filter(np.where(nan_mask, 0, filled), size=5)
        local_cnt  = uniform_filter((~nan_mask).astype(float), size=5)
        filled[nan_mask] = (local_mean / np.maximum(local_cnt, 1e-8))[nan_mask]
    return filled

bathy_f = nanmean_fill(bathy_arr)
backs_f = nanmean_fill(backs_arr)
print('Rasters ready ✓')

Loading raster arrays...
Rasters ready ✓


In [5]:
# ── Terrain derivatives ───────────────────────────────────────────────────────
print('Computing terrain derivatives...')
res          = abs(bath.transform.a)
dz_dy        = sobel(bathy_f, axis=0) / (8 * res)
dz_dx        = sobel(bathy_f, axis=1) / (8 * res)
slope_deg    = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))).astype(np.float32)
aspect_rad   = np.arctan2(dz_dy, -dz_dx).astype(np.float32)
aspect_cos   = np.cos(aspect_rad).astype(np.float32)
aspect_sin   = np.sin(aspect_rad).astype(np.float32)
curvature    = laplace(bathy_f).astype(np.float32) / (res**2)
_kxx         = np.array([[0,0,0],[1,-2,1],[0,0,0]], dtype=np.float32) / res**2
_kyy         = np.array([[0,1,0],[0,-2,0],[0,1,0]], dtype=np.float32) / res**2
plan_curv    = convolve(bathy_f, _kxx).astype(np.float32)
profile_curv = convolve(bathy_f, _kyy).astype(np.float32)
grad_mag     = np.sqrt(dz_dx**2 + dz_dy**2).astype(np.float32)
bpi_inner    = (bathy_f - uniform_filter(bathy_f, size=5)).astype(np.float32)
bpi_broad    = (bathy_f - uniform_filter(bathy_f, size=21)).astype(np.float32)
tri_3        = generic_filter(bathy_f, np.std, size=3).astype(np.float32)
tri_7        = generic_filter(bathy_f, np.std, size=7).astype(np.float32)
rugosity     = (1 + uniform_filter(dz_dx**2 + dz_dy**2, size=7)).astype(np.float32)
tpi_15       = (bathy_f - uniform_filter(bathy_f, size=15)).astype(np.float32)
print('Terrain derivatives done ✓')

Computing terrain derivatives...
Terrain derivatives done ✓


In [6]:
# ── Neighbourhood stats + texture ─────────────────────────────────────────────
print('Computing neighbourhood statistics (2-3 min)...')
WINDOWS = [3, 7, 15, 31]
neighbourhood_rasters = {}

for src_name, src_arr in [('bathy', bathy_f), ('backs', backs_f)]:
    for w in WINDOWS:
        mean_arr  = uniform_filter(src_arr, size=w).astype(np.float32)
        mean_sq   = uniform_filter(src_arr**2, size=w).astype(np.float32)
        std_arr   = np.sqrt(np.maximum(mean_sq - mean_arr**2, 0)).astype(np.float32)
        range_arr = generic_filter(src_arr, np.ptp, size=w).astype(np.float32)
        neighbourhood_rasters[f'{src_name}_mean_w{w}']  = mean_arr
        neighbourhood_rasters[f'{src_name}_std_w{w}']   = std_arr
        neighbourhood_rasters[f'{src_name}_range_w{w}'] = range_arr

backs_q8 = np.round(backs_f / 5).astype(np.int16)
def local_entropy_fn(values):
    vals, cnts = np.unique(values, return_counts=True)
    p = cnts / cnts.sum()
    return -np.sum(p * np.log2(p + 1e-10))

print('  Computing entropy w7...')
neighbourhood_rasters['backs_entropy_w7']  = generic_filter(
    backs_q8.astype(float), local_entropy_fn, size=7).astype(np.float32)
print('  Computing entropy w15...')
neighbourhood_rasters['backs_entropy_w15'] = generic_filter(
    backs_q8.astype(float), local_entropy_fn, size=15).astype(np.float32)
neighbourhood_rasters['backs_sobel'] = np.sqrt(
    sobel(backs_f, axis=0)**2 + sobel(backs_f, axis=1)**2).astype(np.float32)

print(f'Neighbourhood rasters done ✓  ({len(neighbourhood_rasters)} layers)')

Computing neighbourhood statistics (2-3 min)...
  Computing entropy w7...
  Computing entropy w15...
Neighbourhood rasters done ✓  (27 layers)


In [7]:
# ── Sample all rasters at point locations ─────────────────────────────────────
def sample_rasters_at_points(df, transform, feature_dict):
    xs, ys     = df['x'].values, df['y'].values
    rows, cols = rowcol(transform, xs, ys)
    rows = np.clip(rows, 0, bathy_arr.shape[0] - 1)
    cols = np.clip(cols, 0, bathy_arr.shape[1] - 1)
    return pd.DataFrame(
        {name: arr[rows, cols] for name, arr in feature_dict.items()},
        index=df.index
    )

all_rasters = {
    'bathy': bathy_arr, 'backscatter': backs_arr,
    'slope_deg': slope_deg, 'aspect_cos': aspect_cos, 'aspect_sin': aspect_sin,
    'curvature': curvature, 'plan_curv': plan_curv, 'profile_curv': profile_curv,
    'grad_mag': grad_mag, 'bpi_inner': bpi_inner, 'bpi_broad': bpi_broad,
    'tri_3': tri_3, 'tri_7': tri_7, 'rugosity': rugosity, 'tpi_15': tpi_15,
    **neighbourhood_rasters,
}

transform       = bath.transform
train_rich_feat = sample_rasters_at_points(train_df, transform, all_rasters)
test_rich_feat  = sample_rasters_at_points(test_df,  transform, all_rasters)

# Cross-features and spatial
for feat_df, src_df in [(train_rich_feat, train_df), (test_rich_feat, test_df)]:
    feat_df['bathy_x_backs']     = feat_df['bathy']     * feat_df['backscatter']
    feat_df['slope_x_backs']     = feat_df['slope_deg'] * feat_df['backscatter']
    feat_df['tri7_x_backs']      = feat_df['tri_7']     * feat_df['backscatter']
    feat_df['bpi_broad_x_backs'] = feat_df['bpi_broad'] * feat_df['backscatter']
    feat_df['rugosity_x_backs']  = feat_df['rugosity']  * feat_df['backscatter']
    feat_df['backs_per_depth']   = feat_df['backscatter'] / (np.abs(feat_df['bathy']) + 0.01)
    feat_df['tri7_per_slope']    = feat_df['tri_7'] / (feat_df['slope_deg'] + 0.01)
    feat_df['backs_dev_w31']     = feat_df['backscatter'] - feat_df['backs_mean_w31']
    feat_df['bathy_dev_w31']     = feat_df['bathy']       - feat_df['bathy_mean_w31']
    feat_df['x_norm']            = (src_df['x'] - src_df['x'].mean()) / src_df['x'].std()
    feat_df['y_norm']            = (src_df['y'] - src_df['y'].mean()) / src_df['y'].std()
    feat_df['dist_centroid']     = np.sqrt(feat_df['x_norm']**2 + feat_df['y_norm']**2)
    for col in ['tri_3', 'tri_7', 'rugosity', 'grad_mag', 'slope_deg']:
        feat_df[f'log1p_{col}'] = np.log1p(np.abs(feat_df[col]))
    feat_df.fillna(feat_df.median(), inplace=True)

RICH_FEATURES = list(train_rich_feat.columns)
print(f'Rich features: {len(RICH_FEATURES)}')

Rich features: 59


## 4. Label Encoding

In [8]:
CLASSES   = ['SGAM', 'NVB', 'SGZ', 'ALG', 'FMAT']
target    = 'class'
N_CLASSES = len(CLASSES)

le = LabelEncoder()
le.fit(CLASSES)
train_df['label'] = le.transform(train_df[target])
y = train_df['label']

print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'Simple features: {len(SIMPLE_FEATURES)}  |  Rich features: {len(RICH_FEATURES)}')

Label encoding: {np.str_('ALG'): np.int64(0), np.str_('FMAT'): np.int64(1), np.str_('NVB'): np.int64(2), np.str_('SGAM'): np.int64(3), np.str_('SGZ'): np.int64(4)}
Simple features: 26  |  Rich features: 59


## 5. Stage 1 — Rich Meta-Model

Trained on all 50+ rich features from the 6th place notebook.
OOF probabilities are computed via 5-fold CV — no leakage into Stage 2.
For test, predictions are averaged across all 5 folds.

In [9]:
X_rich      = train_rich_feat[RICH_FEATURES]
X_test_rich = test_rich_feat[RICH_FEATURES]

rich_params = {
    'n_estimators':      2000,
    'learning_rate':     0.05,
    'max_depth':         4,
    'num_leaves':        15,
    'subsample':         0.80,
    'colsample_bytree':  0.70,
    'min_child_samples': 5,
    'reg_alpha':         0.5,
    'reg_lambda':        2.0,
    'class_weight':     'balanced',
    'random_state':      42,
    'device':           'gpu',
    'n_jobs':           -1,
    'verbose':          -1,
}

skf_meta           = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_rich_probs     = np.zeros((len(X_rich), N_CLASSES))
test_rich_probs    = np.zeros((len(X_test_rich), N_CLASSES))

print('Training rich meta-model (Stage 1)...')
for fold, (tr_idx, val_idx) in enumerate(skf_meta.split(X_rich, y), 1):
    X_tr, y_tr = X_rich.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X_rich.iloc[val_idx], y.iloc[val_idx]

    m = LGBMClassifier(**rich_params)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[early_stopping(50, verbose=False), log_evaluation(0)]
    )
    oof_rich_probs[val_idx] = m.predict_proba(X_va)
    test_rich_probs        += m.predict_proba(X_test_rich) / skf_meta.n_splits

    fold_f1 = f1_score(y_va, np.argmax(oof_rich_probs[val_idx], axis=1), average='weighted')
    print(f'  Fold {fold}  Rich meta OOF F1: {fold_f1:.5f}')

rich_meta_oof_f1 = f1_score(y, np.argmax(oof_rich_probs, axis=1), average='weighted')
print(f'\nRich meta-model OOF F1: {rich_meta_oof_f1:.5f}')
print('(Reference — main model should beat this by combining with simple features)')

Training rich meta-model (Stage 1)...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  Fold 1  Rich meta OOF F1: 0.98881
  Fold 2  Rich meta OOF F1: 0.98882
  Fold 3  Rich meta OOF F1: 0.98485
  Fold 4  Rich meta OOF F1: 0.97452
  Fold 5  Rich meta OOF F1: 0.98881

Rich meta-model OOF F1: 0.98515
(Reference — main model should beat this by combining with simple features)


## 6. Append Meta-Features to Simple Feature Set

In [10]:
meta_col_names = [f'meta_prob_{cls}' for cls in le.classes_]

# Simple features + 5 meta prob columns
X_simple      = train_df[SIMPLE_FEATURES].copy()
X_test_simple = test_df[SIMPLE_FEATURES].copy()

for i, col in enumerate(meta_col_names):
    X_simple[col]      = oof_rich_probs[:, i]   # OOF — no leakage
    X_test_simple[col] = test_rich_probs[:, i]  # fold-averaged

FINAL_FEATURES = list(X_simple.columns)
print(f'Simple features:              {len(SIMPLE_FEATURES)}')
print(f'+ Meta prob columns:          {len(meta_col_names)}')
print(f'= Total main model features:  {len(FINAL_FEATURES)}')
print(f'Meta columns: {meta_col_names}')

Simple features:              26
+ Meta prob columns:          5
= Total main model features:  31
Meta columns: ['meta_prob_ALG', 'meta_prob_FMAT', 'meta_prob_NVB', 'meta_prob_SGAM', 'meta_prob_SGZ']


## 7. Stage 2 — Main Seed Averaging Model

Same Optuna-tuned LGB + seed averaging as the best-scoring notebook.
Trains on the 26 original simple features + 5 rich meta-prob columns = 31 features total.

The model now has access to what the rich terrain model thinks, while still anchoring its predictions on the same direct observations that made the original model strong.

In [11]:
seeds = [52, 62, 72]

best_params = {
    'n_estimators':      2000,
    'learning_rate':     0.07918500203202344,
    'max_depth':         11,
    'num_leaves':        77,
    'subsample':         0.8055152296793877,
    'colsample_bytree':  0.7103689169741864,
    'reg_alpha':         0.014069050212386091,
    'reg_lambda':        0.7078441464734376,
    'min_child_samples': 9,
    'class_weight':     'balanced',
    'random_state':      42,
    'device':           'gpu',
    'n_jobs':           -1,
    'verbose':          -1,
}

final_test_probs = np.zeros((len(X_test_simple), N_CLASSES))
seed_scores      = []

for seed in seeds:
    print(f'\n===== Seed {seed} =====')
    best_params['random_state'] = seed

    oof_preds   = np.zeros(len(X_simple), dtype=int)
    test_probs  = np.zeros((len(X_test_simple), N_CLASSES))
    skf         = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_simple, y), 1):
        X_tr, y_tr = X_simple.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_simple.iloc[val_idx],   y.iloc[val_idx]

        model = LGBMClassifier(**best_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[early_stopping(50, verbose=False), log_evaluation(0)]
        )

        val_preds          = model.predict(X_va)
        oof_preds[val_idx] = val_preds
        fold_f1            = f1_score(y_va, val_preds, average='weighted')
        fold_scores.append(fold_f1)
        print(f'  Fold {fold} Weighted F1: {fold_f1:.5f}')

        test_probs += model.predict_proba(X_test_simple) / skf.n_splits

    seed_f1 = f1_score(y, oof_preds, average='weighted')
    seed_scores.append(seed_f1)
    print(f'  Seed {seed} OOF Weighted F1: {seed_f1:.5f}')
    print(f'  Seed {seed} Fold Avg F1:     {np.mean(fold_scores):.5f}')

    final_test_probs += test_probs / len(seeds)

print('\n==============================')
print('Seed Scores:', [round(s, 5) for s in seed_scores])
print(f'Main model Avg OOF F1: {np.mean(seed_scores):.5f}')
print(f'Rich meta-model OOF:   {rich_meta_oof_f1:.5f}  (reference)')
print('==============================')


===== Seed 52 =====
  Fold 1 Weighted F1: 0.98644
  Fold 2 Weighted F1: 0.98646
  Fold 3 Weighted F1: 0.99043
  Fold 4 Weighted F1: 0.98641
  Fold 5 Weighted F1: 0.98321
  Seed 52 OOF Weighted F1: 0.98659
  Seed 52 Fold Avg F1:     0.98659

===== Seed 62 =====
  Fold 1 Weighted F1: 0.98557
  Fold 2 Weighted F1: 0.98883
  Fold 3 Weighted F1: 0.98885
  Fold 4 Weighted F1: 0.98322
  Fold 5 Weighted F1: 0.98483
  Seed 62 OOF Weighted F1: 0.98626
  Seed 62 Fold Avg F1:     0.98626

===== Seed 72 =====
  Fold 1 Weighted F1: 0.98484
  Fold 2 Weighted F1: 0.98802
  Fold 3 Weighted F1: 0.98394
  Fold 4 Weighted F1: 0.99121
  Fold 5 Weighted F1: 0.98566
  Seed 72 OOF Weighted F1: 0.98674
  Seed 72 Fold Avg F1:     0.98673

Seed Scores: [0.98659, 0.98626, 0.98674]
Main model Avg OOF F1: 0.98653
Rich meta-model OOF:   0.98515  (reference)


## 8. Generate Submission

In [12]:
final_preds = le.inverse_transform(np.argmax(final_test_probs, axis=1))

submission = pd.DataFrame({
    'ID':   test_df['ID'],
    target: final_preds
})

submission.to_csv('submission_meta_stacked.csv', index=False)
print('Saved: submission_meta_stacked.csv')
print(f'Prediction distribution:\n{pd.Series(final_preds).value_counts()}')

Saved: submission_meta_stacked.csv
Prediction distribution:
NVB     40
ALG     26
FMAT    15
SGAM     9
SGZ      8
Name: count, dtype: int64
